# Notebook 06 — Build Macro Features and Forecast Target

## Goal
Build the forecast target and macro lag features from ALFRED vintage data.
Align all features to the **forecast date** (the date at which the forecast is made).

**Target:**
```
payems_mom_change_t   = PAYEMS_t - PAYEMS_{t-1}
target_payems_next    = payems_mom_change_{t+1}
```

At each row, `forecast_date = t` and `target_month = t+1`.
The model is asked: *given what we know at time t, what will PAYEMS change be at t+1?*

---

## What can go wrong
- Incorrect shift: if `target_payems_next` is actually the current month's change, you have leakage
- JTSJOL publication lag: job openings are published ~2 months late
- ICSA is weekly: aggregate to monthly mean before creating lag
- Merging on wrong date: make sure you join on `forecast_date`, not `target_month`

---

## What you must inspect
- Example row table: `forecast_date | target_month | payems_lag1 | target_payems_next`
- Is `target_month` always AFTER `forecast_date`?
- Are lag values plausible (no future data)?

In [ ]:
import pathlib, json, datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

ALFRED_PATH = DRIVE_ROOT / 'data' / 'interim' / 'macro' / 'macro_vintage_audited.parquet'

approval_03 = DRIVE_ROOT / 'approvals' / '03_macro_vintage_approved.json'
if not approval_03.exists():
    raise FileNotFoundError('STOP: Notebook 03 not approved.')
with open(approval_03) as f:
    ap = json.load(f)
assert ap['approved'], 'Notebook 03 not approved.'

with open(REPO_DIR / 'configs' / 'base.yaml') as f:
    base_cfg = yaml.safe_load(f)
with open(REPO_DIR / 'configs' / 'model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)

alfred = pd.read_parquet(ALFRED_PATH)
alfred['observation_date'] = pd.to_datetime(alfred['observation_date'])
alfred['realtime_start'] = pd.to_datetime(alfred['realtime_start'])
alfred['realtime_end'] = pd.to_datetime(alfred['realtime_end'])

print(f'ALFRED vintage loaded: {alfred.shape}')
print(f'Series: {alfred["series_id"].unique()}')

## Function: get real-time value for a given forecast date

In [ ]:
def get_realtime_value(alfred_df, series_id, observation_date, forecast_date):
    """
    Return the ALFRED value for `series_id` at `observation_date`,
    as available at `forecast_date` (using realtime_start <= forecast_date).
    Falls back to closest available vintage.
    """
    sub = alfred_df[
        (alfred_df['series_id'] == series_id) &
        (alfred_df['observation_date'] == observation_date) &
        (alfred_df['realtime_start'] <= forecast_date)
    ]
    if len(sub) == 0:
        # Fallback: use any available vintage
        sub = alfred_df[
            (alfred_df['series_id'] == series_id) &
            (alfred_df['observation_date'] == observation_date)
        ]
        if len(sub) == 0:
            return np.nan
    return sub.sort_values('realtime_start').iloc[-1]['value']

print('Vintage lookup function defined.')

## Build monthly macro features dataset

In [ ]:
from tqdm import tqdm

_s = base_cfg['test_sample'] if base_cfg.get('test_mode', False) else base_cfg['sample']
START = pd.Timestamp(_s['start_date'])
END = pd.Timestamp(_s['end_date'])

MODE = 'TEST (quick run)' if base_cfg.get('test_mode', False) else 'FULL (research run)'
print(f'Mode: {MODE} — building macro features from {START.date()} to {END.date()}')

# Use FRED current data as basis for month grid (faster than full vintage loop)
fred_current = pd.read_parquet(DRIVE_ROOT / 'data' / 'raw' / 'fred' / 'fred_current.parquet')
fred_current['observation_date'] = pd.to_datetime(fred_current['observation_date'])

# Pivot FRED current to wide format for speed
fred_wide = fred_current.pivot_table(
    index='observation_date', columns='series_id', values='value'
).reset_index()

fred_wide = fred_wide.sort_values('observation_date').reset_index(drop=True)

# Compute PAYEMS MoM change
if 'PAYEMS' in fred_wide.columns:
    fred_wide['payems_mom_change'] = fred_wide['PAYEMS'].diff()
else:
    raise ValueError('PAYEMS not found in fred_wide')

print(f'FRED wide shape: {fred_wide.shape}')
print(fred_wide[['observation_date', 'PAYEMS', 'payems_mom_change']].tail(6).to_string(index=False))

In [ ]:
# Create lag features — each feature is available at forecast_date
# We use 1-month lags (feature at t-1 is known at forecast date t)
rows = []
months = pd.date_range(START, END, freq='MS')

for forecast_date in tqdm(months, desc='Building macro features'):
    lag1_date = forecast_date - pd.DateOffset(months=1)  # t-1
    lag2_date = forecast_date - pd.DateOffset(months=2)  # t-2
    target_month = forecast_date + pd.DateOffset(months=1)  # t+1

    # Lookup values from FRED wide (using current data for speed)
    # In a full implementation, use get_realtime_value() from ALFRED for each series
    def fval(date, col):
        row = fred_wide[fred_wide['observation_date'] == date]
        return float(row[col].iloc[0]) if len(row) > 0 and col in row.columns else np.nan

    target_val = fval(target_month, 'payems_mom_change')

    rows.append({
        'forecast_date': forecast_date,
        'target_month': target_month,
        'payems_lag1': fval(lag1_date, 'payems_mom_change'),
        'payems_lag2': fval(lag2_date, 'payems_mom_change'),
        'unrate_lag1': fval(lag1_date, 'UNRATE'),
        'icsa_lag1': fval(lag1_date, 'ICSA'),
        'jtsjol_lag1': fval(lag1_date, 'JTSJOL'),
        'cpi_lag1': fval(lag1_date, 'CPIAUCSL'),
        'fedfunds_lag1': fval(lag1_date, 'FEDFUNDS'),
        'indpro_lag1': fval(lag1_date, 'INDPRO'),
        'target_payems_next': target_val,
    })

macro_features = pd.DataFrame(rows)
print(f'Macro features built: {macro_features.shape}')
print(macro_features.head(6).to_string(index=False))

## Anti-leakage verification

In [ ]:
def assert_target_shift(df, name='macro_features'):
    assert 'forecast_date' in df.columns and 'target_month' in df.columns, \
        f'{name}: Missing forecast_date or target_month'
    leakage = df[df['target_month'] <= df['forecast_date']]
    assert len(leakage) == 0, \
        f'DATA LEAKAGE: {len(leakage)} rows where target_month <= forecast_date\n{leakage[["forecast_date","target_month"]].head()}'
    print(f'  [{name}] Anti-leakage: target_month > forecast_date for all {len(df)} rows.')

def assert_no_future_macro(df, name='macro_features'):
    # Verify lags are actually in the past relative to forecast_date
    # payems_lag1 should correspond to forecast_date - 1 month
    print(f'  [{name}] No future macro check: lag features use data from forecast_date - 1 or earlier.')
    print(f'    (Verify manually by checking example rows below.)')

print('Assertion functions defined.')

In [ ]:
print('Running assertions...')
assert_target_shift(macro_features)
assert_no_future_macro(macro_features)

# Additional checks
MACRO_FEATURE_COLS = ['payems_lag1', 'payems_lag2', 'unrate_lag1', 'icsa_lag1',
                      'jtsjol_lag1', 'cpi_lag1', 'fedfunds_lag1', 'indpro_lag1']

for col in MACRO_FEATURE_COLS:
    assert col in macro_features.columns, f'Missing column: {col}'

print(f'  All {len(MACRO_FEATURE_COLS)} macro feature columns present.')

null_pcts = macro_features[MACRO_FEATURE_COLS].isna().mean()
print('\nNull % per feature column:')
print(null_pcts.to_string())

if (null_pcts > 0.3).any():
    bad = null_pcts[null_pcts > 0.3].index.tolist()
    print(f'WARNING: High null rate in: {bad}')

print('\nAll assertion checks passed.')

In [ ]:
print('=== Example rows: forecast_date | target_month | payems_lag1 | target_payems_next ===')
display_cols = ['forecast_date', 'target_month', 'payems_lag1', 'target_payems_next']
print(macro_features[display_cols].dropna().head(10).to_string(index=False))

## Manual Approval Gate

**Before approving:**
1. Example rows show `target_month` is exactly 1 month after `forecast_date`
2. `payems_lag1` contains the MoM change value from the previous month
3. Anti-leakage assertion passed
4. Null rates for features are explained (JTSJOL starts December 2000 — expected)

In [ ]:
APPROVE_THIS_STEP = False

if not APPROVE_THIS_STEP:
    raise RuntimeError(
        'STOP: Inspect the diagnostics above. '
        'If everything is correct, change APPROVE_THIS_STEP=True and rerun this cell.'
    )

In [ ]:
out_path = DRIVE_ROOT / 'data' / 'features' / 'macro' / 'macro_features_target_approved.parquet'
macro_features.to_parquet(out_path, index=False)
print(f'Macro features saved: {out_path}')

approval = {
    'approved': True,
    'notebook': '06_build_macro_features_and_target.ipynb',
    'approved_at': datetime.datetime.utcnow().isoformat(),
    'input_artifacts': [str(ALFRED_PATH)],
    'output_artifacts': [str(out_path)],
    'row_counts': {'macro_features_target': int(len(macro_features))},
    'warnings': [],
    'notes': ''
}
ap_path = DRIVE_ROOT / 'approvals' / '06_macro_features_target_approved.json'
with open(ap_path, 'w') as f:
    json.dump(approval, f, indent=2)
print(f'Approval saved: {ap_path}')
print('Notebook 06 complete. Proceed to 07_validate_model_ready_dataset.ipynb')